# Credit Risk Threshold Lab

**Dataset:** UCI Default of Credit Card Clients (29,965 rows after deduplication, 23 features)  
**Positive class:** `default = 1` means the customer defaulted on payment next month  
**Negative class:** `default = 0` means the customer did not default  

This notebook is a guided walkthrough of the full analysis. For a clean pipeline run, use `scripts/run_analysis.py`. The notebook is designed for exploration and explanation.

The central question: **at what probability threshold should we flag a borrower as high-risk, and what does each type of mistake cost?**

## Setup

In [ ]:
import sys
import os
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TARGET_COL, RANDOM_STATE, THRESHOLDS, FP_COST, FN_COST
from src.data import load_uci_credit_dataset, basic_cleaning, split_features_target, train_test_split_data, compute_scale_pos_weight
from src.modeling import build_logistic_regression, build_xgboost, train_model, get_predicted_probabilities
from src.evaluation import evaluate_at_threshold, threshold_sweep, choose_best_threshold, compute_roc_auc, compute_pr_auc, naive_baseline_metrics
from src.plots import (
    plot_class_balance, plot_confusion_matrix, plot_threshold_metrics,
    plot_expected_cost_by_threshold, plot_roc_curve_comparison,
    plot_pr_curve_comparison, plot_feature_importance_xgboost,
)

print("Imports OK")

## 1. Load the UCI dataset

The dataset is fetched automatically via the `ucimlrepo` package. No manual download required. The raw target column is named `Y` and the feature columns are `X1` through `X23`. We rename both to human-readable names.

The dataset contains 30,000 records from Taiwan credit card holders. The target variable is whether the customer defaulted on their October 2005 payment.

In [ ]:
print("Fetching UCI dataset (ID 350)...")
df_raw = load_uci_credit_dataset()
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)

## 2. Cleaning

We check for duplicates and missing values. The dataset is well-formed but contains 35 exact duplicate rows that are dropped.

In [ ]:
df_clean, dupes, nulls = basic_cleaning(df_raw)
print(f"Rows after cleaning: {len(df_clean):,}")
print(f"Duplicates dropped: {dupes}")
print(f"Null rows dropped: {nulls}")
df_clean[TARGET_COL].value_counts()

## 3. Class balance

22.13% of customers in the cleaned dataset defaulted. This imbalance matters: a model that always predicts "no default" achieves 77.87% accuracy and is completely useless for catching defaults.

This is the naive baseline we need to beat meaningfully, not just numerically.

In [ ]:
plot_class_balance(df_clean[TARGET_COL])
plt.show()

counts = df_clean[TARGET_COL].value_counts().sort_index()
pcts = counts / len(df_clean) * 100
print(f"No Default (0): {counts[0]:,}  ({pcts[0]:.2f}%)")
print(f"Default    (1): {counts[1]:,}  ({pcts[1]:.2f}%)")
print(f"Default rate: {df_clean[TARGET_COL].mean():.4f}")

## 4. Train/test split

We use a stratified 80/20 split so the default rate is preserved in both train and test sets.

In [ ]:
X, y = split_features_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

scale_pos_weight = compute_scale_pos_weight(y_train)

print(f"Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")
print(f"Train default rate: {y_train.mean():.4f}")
print(f"Test  default rate: {y_test.mean():.4f}")
print(f"XGBoost scale_pos_weight: {scale_pos_weight}")
print("  (computed as non-defaulters / defaulters in training data)")

## 5. Why accuracy is not enough

Before training any model, let's make the problem concrete. The naive baseline -- predicting no default for everyone -- achieves 77.87% test accuracy. That is the bar. A model that scores below this on accuracy is not necessarily worse; it may simply be predicting more defaults and catching the ones that matter.

In [ ]:
from sklearn.metrics import accuracy_score
naive_acc = (y_test == 0).mean()
print(f"Naive baseline accuracy (always predict 0): {naive_acc:.4f}")
print()
print("A model trained to catch defaults will typically score below this on accuracy.")
print("That does not mean it is worse. It means accuracy is the wrong scorecard.")

## 6. Train logistic regression

We use `class_weight='balanced'` to compensate for the 22% / 78% imbalance. This adjusts the training loss so misclassifying a defaulter costs more than misclassifying a non-defaulter, proportional to the class frequencies. A StandardScaler is applied first since logistic regression is sensitive to feature scale.

In [ ]:
lr = build_logistic_regression()
lr = train_model(lr, X_train, y_train)
lr_proba = get_predicted_probabilities(lr, X_test)

print(f"Logistic Regression trained.")
print(f"Predicted probability range: [{lr_proba.min():.4f}, {lr_proba.max():.4f}]")

## 7. Train XGBoost

XGBoost uses gradient boosted trees with shallow depth (max_depth=3) and a low learning rate (0.05). No scaling is applied since tree-based models are not sensitive to feature magnitudes. The `scale_pos_weight` parameter performs the same role as balanced class weights: it up-weights the minority class during training.

In [ ]:
xgb = build_xgboost(scale_pos_weight=scale_pos_weight)
xgb = train_model(xgb, X_train, y_train)
xgb_proba = get_predicted_probabilities(xgb, X_test)

print(f"XGBoost trained. scale_pos_weight={scale_pos_weight}")
print(f"Predicted probability range: [{xgb_proba.min():.4f}, {xgb_proba.max():.4f}]")

## 8. Evaluate at threshold 0.50

The 0.50 threshold is rarely optimal for imbalanced problems, but it is the standard comparison point. We show all key metrics here before moving to the sweep.

In [ ]:
naive = naive_baseline_metrics(y_test)
lr_50 = evaluate_at_threshold(y_test, lr_proba, 0.50)
xgb_50 = evaluate_at_threshold(y_test, xgb_proba, 0.50)

lr_roc = compute_roc_auc(y_test, lr_proba)
xgb_roc = compute_roc_auc(y_test, xgb_proba)
lr_pr = compute_pr_auc(y_test, lr_proba)
xgb_pr = compute_pr_auc(y_test, xgb_proba)

summary = pd.DataFrame([
    {"Model": "Naive (predict 0)", "Accuracy": naive["accuracy"], "Precision": naive["precision"],
     "Recall": naive["recall"], "F1": naive["f1"], "ROC-AUC": 0.0, "PR-AUC": 0.0},
    {"Model": "Logistic Regression", "Accuracy": lr_50["accuracy"], "Precision": lr_50["precision"],
     "Recall": lr_50["recall"], "F1": lr_50["f1"], "ROC-AUC": lr_roc, "PR-AUC": lr_pr},
    {"Model": "XGBoost", "Accuracy": xgb_50["accuracy"], "Precision": xgb_50["precision"],
     "Recall": xgb_50["recall"], "F1": xgb_50["f1"], "ROC-AUC": xgb_roc, "PR-AUC": xgb_pr},
])

display(summary.round(4))
print()
print("Note: LR accuracy at t=0.50 falls below naive baseline because balanced class weights")
print("push the model toward predicting more defaults, which reduces accuracy but improves recall.")

In [ ]:
lr_pred_50 = (lr_proba >= 0.50).astype(int)
xgb_pred_50 = (xgb_proba >= 0.50).astype(int)

plot_confusion_matrix(y_test, lr_pred_50,
                      title="Logistic Regression -- Threshold 0.50",
                      filename="logistic_confusion_matrix_t050.png")
plot_confusion_matrix(y_test, xgb_pred_50,
                      title="XGBoost -- Threshold 0.50",
                      filename="xgboost_confusion_matrix_t050.png")

## 9. ROC and Precision-Recall curves

**ROC-AUC** measures the model's ability to rank defaulters above non-defaulters across all thresholds. Higher is better. XGBoost (0.7757) outperforms logistic regression (0.7162).

**PR-AUC** (average precision) measures precision-recall tradeoff at each threshold. For imbalanced datasets, PR-AUC is the more informative metric because ROC can look favorable even when the model is making many false positives in absolute terms. A no-skill baseline would achieve a PR-AUC equal to the class prevalence (0.2213 here).

In [ ]:
proba_dict = {
    "Logistic Regression": lr_proba,
    "XGBoost": xgb_proba,
}

plot_roc_curve_comparison(y_test, proba_dict)
plot_pr_curve_comparison(y_test, proba_dict)
print(f"LR  ROC-AUC: {lr_roc:.4f}  |  PR-AUC: {lr_pr:.4f}")
print(f"XGB ROC-AUC: {xgb_roc:.4f}  |  PR-AUC: {xgb_pr:.4f}")

## 10. Threshold sweep

We evaluate both models at 19 thresholds from 0.05 to 0.95. The sweep shows how precision, recall, and F1 trade off as we move the decision boundary.

Lower threshold: more borrowers flagged as risky, higher recall, lower precision.  
Higher threshold: fewer flagged, higher precision, lower recall.

Neither direction is inherently better. The right threshold depends on what each type of mistake costs.

In [ ]:
lr_sweep = threshold_sweep(y_test, lr_proba, THRESHOLDS, fp_cost=FP_COST, fn_cost=FN_COST)
xgb_sweep = threshold_sweep(y_test, xgb_proba, THRESHOLDS, fp_cost=FP_COST, fn_cost=FN_COST)

plot_threshold_metrics(lr_sweep, model_name="Logistic Regression")
plot_threshold_metrics(xgb_sweep, model_name="XGBoost")

print("XGBoost threshold sweep (key columns):")
display(xgb_sweep[["threshold", "precision", "recall", "f1",
                    "false_positives", "false_negatives", "expected_cost"]].round(4))

## 11. Business cost assumptions

We now attach dollar costs to each type of error.

**These are example assumptions for this exercise only.** They are not based on real loan amounts, interest rates, or operational costs. Before using this framework operationally, the cost figures must be estimated from actual portfolio data with input from credit risk and finance teams.

- **False positive cost: $500** -- We flag a creditworthy borrower as risky. They may be denied or offered worse terms. The lender loses a good customer.
- **False negative cost: $5,000** -- We approve a borrower who will default. The lender absorbs the loss.

Expected cost = (false positives x $500) + (false negatives x $5,000)

In [ ]:
print(f"False positive cost: ${FP_COST:,}  (illustrative)")
print(f"False negative cost: ${FN_COST:,}  (illustrative)")
print(f"Cost ratio: {FN_COST // FP_COST}:1 (FN is {FN_COST // FP_COST}x more costly)")

## 12. Select cost-minimizing threshold

In [ ]:
plot_expected_cost_by_threshold(lr_sweep, xgb_sweep, FP_COST, FN_COST)

lr_best = choose_best_threshold(lr_sweep)
xgb_best = choose_best_threshold(xgb_sweep)

print(f"Logistic Regression:")
print(f"  Best threshold:   {lr_best['threshold']}")
print(f"  Precision:        {lr_best['precision']:.4f}")
print(f"  Recall:           {lr_best['recall']:.4f}")
print(f"  F1:               {lr_best['f1']:.4f}")
print(f"  False positives:  {int(lr_best['false_positives'])}")
print(f"  False negatives:  {int(lr_best['false_negatives'])}")
print(f"  Expected cost:    ${lr_best['expected_cost']:,.0f}")
print()
print(f"XGBoost:")
print(f"  Best threshold:   {xgb_best['threshold']}")
print(f"  Precision:        {xgb_best['precision']:.4f}")
print(f"  Recall:           {xgb_best['recall']:.4f}")
print(f"  F1:               {xgb_best['f1']:.4f}")
print(f"  False positives:  {int(xgb_best['false_positives'])}")
print(f"  False negatives:  {int(xgb_best['false_negatives'])}")
print(f"  Expected cost:    ${xgb_best['expected_cost']:,.0f}")

## 13. Final recommendation and confusion matrix

In [ ]:
# XGBoost at 0.20 has lower expected cost
winner_name = "XGBoost"
winner_threshold = xgb_best["threshold"]
winner_proba = xgb_proba

final_pred = (winner_proba >= winner_threshold).astype(int)
plot_confusion_matrix(
    y_test, final_pred,
    title=f"{winner_name} at Threshold {winner_threshold} (Recommended)",
    filename="final_model_confusion_matrix.png"
)

print(f"FINAL RECOMMENDATION: {winner_name} at threshold {winner_threshold}")
print()
print(f"On test set ({len(y_test):,} customers):")
print(f"  Actual defaults:            {int(y_test.sum())}")
print(f"  Defaults correctly caught:  {int(xgb_best['true_positives'])} ({xgb_best['recall']*100:.1f}%)")
print(f"  Defaults missed:            {int(xgb_best['false_negatives'])}")
print(f"  Non-default customers incorrectly flagged: {int(xgb_best['false_positives'])}")
print(f"  Expected cost: ${xgb_best['expected_cost']:,.0f}  (illustrative)")

## 14. Feature importance

**XGBoost feature importance by gain:** PAY_0 (most recent repayment status) accounts for 35.3% of total gain, far ahead of all other features. This is consistent with the intuition that recent payment behavior is the strongest predictor of imminent default.

**Logistic regression coefficients:** Larger positive coefficients are associated with higher predicted default probability. Larger negative coefficients are associated with lower predicted default probability. These are associations in the model, not causal claims.

In [ ]:
plot_feature_importance_xgboost(xgb, list(X_train.columns), top_n=15)

fi_df = pd.DataFrame({
    "feature": list(X_train.columns),
    "importance": xgb.feature_importances_,
}).sort_values("importance", ascending=False)

print("Top 10 XGBoost features:")
display(fi_df.head(10).reset_index(drop=True))

# Logistic regression coefficients
lr_clf = lr.named_steps["clf"]
coef_df = pd.DataFrame({
    "feature": list(X_train.columns),
    "coefficient": lr_clf.coef_[0],
}).sort_values("coefficient", ascending=False)

print()
print("Logistic Regression -- top 5 (associated with higher default risk):")
display(coef_df.head(5).reset_index(drop=True))
print("Logistic Regression -- bottom 5 (associated with lower default risk):")
display(coef_df.tail(5).reset_index(drop=True))

## 15. SHAP (optional)

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, show=True)
except ImportError:
    print("shap not installed. Run: pip install shap")
except Exception as e:
    print(f"SHAP plot skipped: {e}")
    print("Feature importance plot is used as the primary interpretability tool.")

## 16. Key takeaways

**Accuracy is the wrong scorecard.** The naive baseline achieves 77.87% accuracy by predicting no defaults and catching zero. Logistic regression at t=0.50 achieves 68.65% -- below the naive baseline -- while catching 63% of actual defaults. That is not a failure; it is the model doing its job at a cost of reduced accuracy.

**The threshold is a business decision, not a model hyperparameter.** Moving from t=0.50 to t=0.20 for XGBoost increases recall from 63.2% to 96.7% while reducing precision from 45.7% to 25.2%. Whether that tradeoff is right depends on the actual cost ratio between missed defaults and false alarms -- a question that belongs to the credit risk and finance teams.

**PR-AUC is more informative than ROC-AUC for imbalanced data.** XGBoost PR-AUC (0.5557) is more than twice the no-skill baseline (0.2213). This measures how well the model performs across all operating points on the minority class specifically.

**Recent payment behavior dominates.** PAY_0 alone accounts for 35.3% of XGBoost's feature importance. The model is primarily using recent payment status, which is the most intuitive and defensible signal for short-term default prediction.

**The cost curve is flat around the optimum.** XGBoost at t=0.20 has expected cost $2,127,500 and at t=0.25 it is $2,130,500. This $3,000 difference is within noise -- the true optimum is not a single point but a region. Operational considerations (review capacity, fairness constraints) should drive the final choice within that region.